In [ ]:
import numpy as np
import h5py
import lab
import potcorr
import math
import matplotlib.pyplot as plt
import const
import calcmat

In [ ]:
test_dir = '/anvil/scratch/x-rg47749/mos2/unit/sg15/mos2.save'
chg_file = test_dir+'/charge-density.hdf5'
wfc_file = test_dir+'/wfc1.hdf5'
wfc2_file = test_dir+'/wfc2.hdf5'
ibnd_v = 13

cell=lab.mos2_unit_tt
ttkw = potcorr.PotCorr(cell)
ttkw.fft_init()


In [ ]:
pert = potcorr.PotCorr(lab.mos2_6x6)
pert.fft_init()

In [ ]:
pert2 = potcorr.PotCorr(lab.mos2_12to48)
pert2.fft_init()

In [ ]:
A_uc = pert2.lattpara_unit[0]*pert2.lattpara_unit[0]/const.Bohr_R/const.Bohr_R*np.sqrt(3)/2

print(A_uc)

In [ ]:
import utli
pot_d = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/d-relax/Pot.dat")
pot_p = utli.read_dat("/anvil/projects/x-che190065/rjguo/mos2/potential/dft/6x6/p/Pot.dat")
dpot = pot_d-pot_p

In [ ]:
dpot = pot_d-pot_p

dpot2 = np.zeros_like(dpot)

nx, ny, nz= dpot2.shape
dpot2[:nx//2,:nx//2,:]=dpot[nx//2:,nx//2:,:]
dpot2[:nx//2,-nx//2:,:]=dpot[nx//2:,:nx//2,:]
dpot2[-nx//2:,:nx//2,:]=dpot[:nx//2,-nx//2:,:]
dpot2[-nx//2:,-nx//2:,:]=dpot[:nx//2,:nx//2,:]


fine_data = dpot2
f_transform = np.fft.fftn(fine_data)
nx_, ny_, nz_ = fine_data.shape

new_f_transform = np.zeros((nx_//2,ny_//2,nz_), dtype=complex)
nx, ny, nz = new_f_transform.shape

new_f_transform[:nx//2, :ny//2, :nz//2] = f_transform[:nx//2, :ny//2, :nz//2]
new_f_transform[-nx//2:, :ny//2, :nz//2] = f_transform[-nx//2:, :ny//2, :nz//2]
new_f_transform[:nx//2, -ny//2:, :nz//2] = f_transform[:nx//2, -ny//2:, :nz//2]
new_f_transform[-nx//2:, -ny//2:, :nz//2] = f_transform[-nx//2:, -ny//2:, :nz//2]
new_f_transform[:nx//2, :ny//2, -nz//2:] = f_transform[:nx//2, :ny//2, -nz//2:]
new_f_transform[-nx//2:, :ny//2, -nz//2:] = f_transform[-nx//2:, :ny//2, -nz//2:]
new_f_transform[:nx//2, -ny//2:, -nz//2:] = f_transform[:nx//2, -ny//2:, -nz//2:]
new_f_transform[-nx//2:, -ny//2:, -nz//2:] = f_transform[-nx//2:, -ny//2:, -nz//2:]


fine_data2 = np.fft.ifftn(new_f_transform).real/4

dpot_l = np.zeros((720,720,225), dtype=complex )

dpot_l[:nx//2, :nx//2, :] = fine_data2[:nx//2, :nx//2, :]
dpot_l[:nx//2, -nx//2:, :] = fine_data2[:nx//2, -nx//2:, :]
dpot_l[-nx//2:, :nx//2, :] = fine_data2[-nx//2:, :nx//2, :]
dpot_l[-nx//2:, -nx//2:, :] = fine_data2[-nx//2:, -nx//2:, :]




In [ ]:
pot_k_z = np.zeros([720,720,225],dtype=complex)
A_uc = pert2.lattpara_unit[0]*pert2.lattpara_unit[0]/const.Bohr_R/const.Bohr_R*np.sqrt(3)/2
for i in range(225):
    pot_k_z[:,:,i] = np.fft.fftn(dpot_l[:,:,i])*pert2.lattpara[0]*pert2.lattpara[1]/720/720*np.sqrt(3)/2 / A_uc

In [ ]:

pot_r_z = np.zeros([720,720,225])
for i in range(225):
    pot_r_z[:,:,i] = np.fft.ifftn(pot_k_z[:,:,i])

In [ ]:
#plt.plot(np.sum(pot_r_z, axis=(0,1)))
plt.plot(np.sum(dpot_l, axis=(0,1))*pert2.lattpara[0]*pert2.lattpara[1]*np.sqrt(3)/2 /720/720)

plt.plot(np.sum(dpot, axis=(0,1))*pert.lattpara[0]*pert.lattpara[1]*np.sqrt(3)/2/180/180)

In [ ]:
np.save('/anvil/projects/x-che190065/rjguo/mos2/potential/standard/neutral/pot_k_z.npy',pot_k_z[:,:,225//2-40: 225//2+40] )

In [ ]:
wfc_file = test_dir+'/wfc13.hdf5'
wfc_r = calcmat.get_wfc_r(wfc_file, ttkw, ibnd_v)

In [ ]:
wfc_sc_r = np.tile(wfc_r, (6, 6, 1))

In [ ]:
x_plot = (pert.fft_xx[:,:,pert.fft_nz//2]-0.5*pert.fft_yy[:,:,pert.fft_nz//2]).flatten()
y_plot = (pert.fft_yy[:,:,pert.fft_nz//2]*np.sqrt(3)/2).flatten()
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
hb1 = ax.scatter(x_plot, y_plot, c=dpot[:,:,225//2+20].real , s=0.1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (pert2.fft_xx[:,:,pert2.fft_nz//2]-0.5*pert2.fft_yy[:,:,pert2.fft_nz//2]).flatten()
y_plot = (pert2.fft_yy[:,:,pert2.fft_nz//2]*np.sqrt(3)/2).flatten()
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
hb1 = ax.scatter(x_plot, y_plot, c=dpot_l[:,:,225//2+20].real , s=0.1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (pert.fft_xx[:,:,pert.fft_nz//2]-0.5*pert.fft_yy[:,:,pert.fft_nz//2]).flatten()
y_plot = (pert.fft_yy[:,:,pert.fft_nz//2]*np.sqrt(3)/2).flatten()
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
fig, ax = plt.subplots(figsize=(8,8),dpi=100)
hb1 = ax.scatter(x_plot, y_plot, c=wfc_sc_r[:,:,225//2].real , s=0.1,
                #gridsize=100,
               # vmax=0.05,
               # vmin=0.000,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
plt.plot(np.sum(wfc_sc_r*wfc_sc_r.conjugate()*dpot, axis=(0,1))[225//2-40:225//2+40]/180/180*pert.lattpara[0]*pert.lattpara[1]*np.sqrt(3)/2)
print(np.sum(wfc_sc_r*wfc_sc_r.conjugate()*dpot)/180/180/225*pert.lattpara[0]*pert.lattpara[1]*pert.lattpara[2]*np.sqrt(3)/2)

In [ ]:

import numpy as np
import h5py
import lab
import potcorr
import math
import matplotlib.pyplot as plt
import const
import calcmat
import pickle

#with open(f'/anvil/projects/x-che190065/rjguo/mos2/potential/standard/neutral/pertpot_k_rbf_model_real.pkl', 'rb') as file:
#    pertpot_k_interp_real = pickle.load(file)

#with open(f'/anvil/projects/x-che190065/rjguo/mos2/potential/standard/neutral/pertpot_k_rbf_model_imag.pkl', 'rb') as file:
#    pertpot_k_interp_imag = pickle.load(file)


with open(f'/anvil/projects/x-che190065/rjguo/mos2/potential/standard/12x12/1e13_dl/pertpot_k_rbf_model_real.pkl', 'rb') as file:
    pertpot_k_interp_real = pickle.load(file)
with open(f'/anvil/projects/x-che190065/rjguo/mos2/potential/standard/12x12/1e13_dl/pertpot_k_rbf_model_imag.pkl', 'rb') as file:
    pertpot_k_interp_imag = pickle.load(file)

In [ ]:
ttkw_wfc = potcorr.PotCorr(lab.mos2_unit_new)
ttkw_wfc.fft_init()


In [ ]:
import scipy.sparse as sp

wmat_kikf = sp.load_npz('/home/x-rg47749/code/triangle/output/wmat_kikf_300x300.npz').tolil()
rows, cols = wmat_kikf.nonzero()
all_indices = set(rows) | set(cols)
#all_indices = set(cols)
print("Number of wavefunctions:",len(all_indices))
print("Number of matrices:",len(wmat_kikf.nonzero()[0]))
print("\n Initializting")


sorted_ind_list = sorted(all_indices)
wfc_ind_qe2wan={}
wfc_ind_wan2qe={}
for i,j in enumerate(sorted_ind_list):
    wfc_ind_qe2wan[i+1]=j
    wfc_ind_wan2qe[j]=i+1

k_pair_dict={}
for i, j in enumerate(zip(rows, cols)):
    k_pair_dict[i]=j



Gxx, Gyy = np.meshgrid(ttkw_wfc.fft_kx_ttt, ttkw_wfc.fft_ky_ttt, indexing='ij')
Gxx = Gxx.flatten()
Gyy = Gyy.flatten()
nkf =300

frac2cart_mat = np.array([[1,0],[np.sqrt(3)/3, 2*np.sqrt(3)/3]])
wfc_dir = '/anvil/projects/x-che190065/rjguo/mos2/wfc_0325/mos2.save'

A_uc_sqrt = np.sqrt(ttkw_wfc.lattpara_unit[0]*ttkw_wfc.lattpara_unit[0]/const.Bohr_R/const.Bohr_R*np.sqrt(3)/2)
pert_nz = 80
Lz = ttkw_wfc.lattpara_unit[2]/const.Bohr_R*pert_nz/ttkw_wfc.fft_nz
b_para = 2*np.pi/(ttkw_wfc.lattpara_unit[0]/const.Bohr_R)

kpid=9

mat_ = calcmat.get_matelem_kp_ks_npy_2(kp_ind=kpid, k_pair_dict=k_pair_dict, wfc_ind_wan2qe=wfc_ind_wan2qe,
                                nkf=300, wfc_dir=wfc_dir, pertpot_k_interp_real=pertpot_k_interp_real,pertpot_k_interp_imag=pertpot_k_interp_imag,
                                Gxx=Gxx, Gyy=Gyy, A_uc_sqrt=A_uc_sqrt,
                                frac2cart_mat=frac2cart_mat,b_para=b_para,Lz=Lz)

#olp_ = calcmat.get_wfc_overlap_kp_ks_npy(kp_ind=kpid, k_pair_dict=k_pair_dict, wfc_ind_wan2qe=wfc_ind_wan2qe, 
#                                         nkf=300, wfc_dir=wfc_dir,Gxx=Gxx, Gyy=Gyy, A_uc_sqrt=A_uc_sqrt,
#                                         frac2cart_mat=frac2cart_mat,b_para=b_para,Lz=Lz)

In [ ]:
A_uc = ttkw_wfc.lattpara_unit[0]*ttkw_wfc.lattpara_unit[0]/const.Bohr_R/const.Bohr_R*np.sqrt(3)/2
print(A_uc)

In [ ]:
kp_ind=0

kp, _, kp_delta_k=calcmat.k_pair_info(kp_ind=kp_ind, k_pair_dict=k_pair_dict, nkf=300)
kiwan, kfwan = kp
kiqe, kfqe = wfc_ind_wan2qe[kiwan], wfc_ind_wan2qe[kfwan]
print(kiqe, kfqe)
_

In [ ]:
plt.plot(mat_)
#plt.plot(olp_*Lz/pert_nz)
print(np.sum(mat_*Lz/pert_nz))
#print(np.sum(olp_*Lz/pert_nz))

In [ ]:
plt.plot(np.sum(wfc_r_npy*wfc_r_npy.conjugate(), axis=(0,1))*ttkw_wfc.lattpara[0]*ttkw_wfc.lattpara[1]/15/15*np.sqrt(3)/2)
plt.plot(olp_)
#plt.ylim(-1,8)

In [ ]:
wfc_r_npy = np.load('/anvil/projects/x-che190065/rjguo/mos2/wfc_0325/mos2.save/wfcr_13.npy')
print(np.sum(wfc_r_npy*wfc_r_npy.conjugate())*ttkw_wfc.lattpara[0]*ttkw_wfc.lattpara[1]*ttkw_wfc.lattpara[2]/15/15/225*np.sqrt(3)/2)

wfc_sc_r_npy = np.tile(wfc_r_npy, (48, 48, 1))

In [ ]:
#plt.plot(np.sum(dpot_l, axis=(0,1))[225//2-40:225//2+40])
#plt.plot(np.sum(wfc_sc_r_npy*wfc_sc_r_npy.conjugate(), axis=(0,1))*pert2.lattpara[0]*pert2.lattpara[1]*np.sqrt(3)/2/720/720)
plt.plot(np.sum(wfc_sc_r_npy*wfc_sc_r_npy.conjugate()*dpot_l[:,:,225//2-40:225//2+40], axis=(0,1))*pert2.lattpara[0]*pert2.lattpara[1]*np.sqrt(3)/2/720/720)

In [ ]:
nx_, ny_, nz_ = ttkw.fft_nx,  ttkw.fft_ny,  ttkw.fft_nz

p_r =  (wfc_r.conj()*wfc_r).real#*nx_*ny_*nz_

fig,ax = plt.subplots(figsize = (8,6), dpi = 100)
#ax.plot(range(nz_), np.sum(wfc_r.real, axis=(0,1)))
#ax.plot(range(nz_), np.sum(wfc_r.imag, axis=(0,1)))
#ax.plot(range(nz_), np.sum(np.abs(wfc_r) , axis=(0,1)))
ax.plot(np.array(range(nz_))/225*2.4, np.sum(np.abs(p_r) , axis=(0,1)))
plt.show()


In [ ]:
xi_z = np.sum(np.abs(p_r) , axis=(0,1))

In [ ]:
xi_Gz = np.fft.fftn(xi_z)

In [ ]:
fig,ax = plt.subplots(figsize = (8,6), dpi = 100)
#ax.plot(range(nz_), np.sum(wfc_r.real, axis=(0,1)))
#ax.plot(range(nz_), np.sum(wfc_r.imag, axis=(0,1)))
#ax.plot(range(nz_), np.sum(np.abs(wfc_r) , axis=(0,1)))
ax.plot(ttkw.fft_kz_ttt, xi_Gz.real)
plt.show()

In [ ]:
np.save('/anvil/scratch/x-rg47749/data/mos2/xi/xi_z', xi_z)

In [ ]:
np.save('/anvil/scratch/x-rg47749/data/mos2/xi/xi_Gz', xi_Gz)

In [ ]:
xi_Gz_try = np.zeros_like(xi_Gz)
xi_Gz_try[0:20] = xi_Gz[0:20]
xi_Gz_try[205:] = xi_Gz[205:]
xi_z_try = np.fft.ifftn(xi_Gz_try)

fig,ax = plt.subplots(figsize = (8,6), dpi = 100)
#ax.plot(range(nz_), np.sum(wfc_r.real, axis=(0,1)))
#ax.plot(range(nz_), np.sum(wfc_r.imag, axis=(0,1)))
#ax.plot(range(nz_), np.sum(np.abs(wfc_r) , axis=(0,1)))
ax.plot(np.array(range(nz_))/225*2.4, xi_z_try.real)
plt.show()

fig,ax = plt.subplots(figsize = (8,6), dpi = 100)
#ax.plot(range(nz_), np.sum(wfc_r.real, axis=(0,1)))
#ax.plot(range(nz_), np.sum(wfc_r.imag, axis=(0,1)))
#ax.plot(range(nz_), np.sum(np.abs(wfc_r) , axis=(0,1)))
ax.plot(ttkw.fft_kz_ttt, xi_Gz_try.real)
plt.show()

In [ ]:
ttkw.fft_kz_ttt

In [ ]:
nx_, ny_, nz_ = ttkw.fft_nx,  ttkw.fft_ny,  ttkw.fft_nz

p_r =  (wfc_r.conj()*wfc_r).real*nx_*ny_*nz_

fig,ax = plt.subplots(figsize = (8,6), dpi = 100)
ax.plot(range(nz_), np.sum(wfc_r.real, axis=(0,1)))
ax.plot(range(nz_), np.sum(wfc_r.imag, axis=(0,1)))
ax.plot(range(nz_), np.sum(np.abs(wfc_r) , axis=(0,1)))
ax.plot(range(nz_), np.sum(np.abs(p_r) , axis=(0,1)))
plt.show()

p2_r =  (wfc2_r.conj()*wfc2_r).real*nx_*ny_*nz_

fig,ax = plt.subplots(figsize = (8,6), dpi = 100)
ax.plot(range(nz_), np.sum(wfc2_r.real, axis=(0,1)))
ax.plot(range(nz_), np.sum(wfc2_r.imag, axis=(0,1)))
ax.plot(range(nz_), np.sum(np.abs(wfc2_r) , axis=(0,1)))
ax.plot(range(nz_), np.sum(np.abs(p2_r) , axis=(0,1)))
plt.show()




In [ ]:
kpts = np.loadtxt('kpt_test.dat')

In [ ]:
np.sum((wfc_sc_r*wfc_sc_r.conj())[:,:,ttkw.fft_nz//2-50:ttkw.fft_nz//2+50])

In [ ]:
wfc_r = calcmat.get_wfc_r_npy(wfc_file)
wfc2_r = calcmat.get_wfc_r_npy(wfc2_file)

In [ ]:
delta_k = (kpts[0]-kpts[1])*2*np.pi/ttkw.lattpara_unit[0]/const.Bohr_R

In [ ]:
phase_r = calcmat.get_phase_r(delta_k, pert)

In [ ]:
wfc_sc_r = np.tile(wfc_r, (24, 24, 1))/24
wfc2_sc_r = np.tile(wfc2_r, (24, 24, 1))/24

In [ ]:
pert_pot=np.load('/anvil/scratch/x-rg47749/data/mos2/12x12/pot_tot_by_wcoul_12-24.npy')

In [ ]:
x_plot = (pert.fft_xx[:,:,pert.fft_nz//2]-0.5*pert.fft_yy[:,:,pert.fft_nz//2]).flatten()
y_plot = (pert.fft_yy[:,:,pert.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(12,12),dpi=200)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=phase_r.real[:,:,pert.fft_nz//2], 
                #gridsize=100,
                #vmax=0.07,
                #vmin=0.0,
                 s=1,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (pert.fft_xx[:,:,pert.fft_nz//2]-0.5*pert.fft_yy[:,:,pert.fft_nz//2]).flatten()
y_plot = (pert.fft_yy[:,:,pert.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(12,12),dpi=200)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=(wfc2_sc_r.conj()*wfc_sc_r).real[:,:,pert.fft_nz//2], 
                #gridsize=100,
                #vmax=0.07,
                #vmin=0.0,
                    s=0.1,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (pert.fft_xx[:,:,pert.fft_nz//2]-0.5*pert.fft_yy[:,:,pert.fft_nz//2]).flatten()
y_plot = (pert.fft_yy[:,:,pert.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(12,12),dpi=200)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=wfc_sc_r.real[:,:,pert.fft_nz//2], 
                #gridsize=100,
                #vmax=0.07,
                #vmin=0.0,
                    s=0.1,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
x_plot = (pert.fft_xx[:,:,pert.fft_nz//2]-0.5*pert.fft_yy[:,:,pert.fft_nz//2]).flatten()
y_plot = (pert.fft_yy[:,:,pert.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(12,12),dpi=200)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=pert_pot.real[:,:,pert.fft_nz//2], 
                #gridsize=100,
                #vmax=0.07,
                #vmin=0.0,
                    s=0.1,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
mat_r = wfc2_sc_r.conj()*wfc_sc_r*phase_r*pert_pot

In [ ]:
x_plot = (pert.fft_xx[:,:,pert.fft_nz//2]-0.5*pert.fft_yy[:,:,pert.fft_nz//2]).flatten()
y_plot = (pert.fft_yy[:,:,pert.fft_nz//2]*np.sqrt(3)/2).flatten()
fig, ax = plt.subplots(figsize=(12,12),dpi=200)
#hb1 = ax.scatter(x_plot, y_plot, c=np.sum(pot_tot_r.real[:,:,ttkw.fft_nz//2-1:ttkw.fft_nz//2+1],axis=(2)), 
hb1 = ax.scatter(x_plot, y_plot, c=mat_r.real[:,:,pert.fft_nz//2], 
                #gridsize=100,
                #vmax=0.07,
                #vmin=0.0,
                    s=0.1,
                    cmap='coolwarm')
ax.set_aspect('equal', adjustable='box')
#ax.set_xlim(0, 180)
#ax.set_ylim(0, 180)
cb1 = fig.colorbar(hb1, ax=ax,fraction = 0.035)
plt.show()

In [ ]:
np.sum(mat_r[:,:,ttkw.fft_nz//2-50:ttkw.fft_nz//2+50])

In [ ]:
np.sum(mat_r)

In [ ]:
np.sum((wfc2_sc_r.conj()*wfc_sc_r)[:,:,])